# Online Model Search

Walk-forward experiments for flow and price models on persisted market-maker interval data.

This notebook is intentionally separate from `production_models_walkforward.ipynb`. It compares production baselines, hyperparameter sweeps, and research-only candidate models under the same prequential contract:

1. Score step `t` using parameters learned only through `t-1`.
2. Update the model with the transition into `t`.
3. Compare models by one-step-forward NLL within the same family.

In [ ]:
from __future__ import annotations

import json
import math
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "README.md").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from mm.env_loader import load_mm_env
from research.model_lab.core.analysis import (
    leaderboard,
    paired_nll_deltas,
    rolling_mean_nll,
    stationarity_report,
    stratified_nll,
)
from research.model_lab.core.walk_forward import run_walk_forward
from research.model_lab.data.loader import load_interval_steps, parse_since
from research.model_lab.models.registry import build_model, list_builtin_models

load_mm_env(REPO_ROOT / ".env")

try:
    import pandas as pd
except Exception:  # pragma: no cover - notebook convenience
    pd = None

try:
    import matplotlib.pyplot as plt
except Exception:  # pragma: no cover - notebook convenience
    plt = None

print(f"repo: {REPO_ROOT}")
print(f"registered models: {len(list_builtin_models())}")

In [ ]:
DATA_CONFIG = {
    "exchange": "nobitex",
    "symbol": "USDT_IRT",
    "interval_ms": 30_000,
    "since": "48h",
    "until": None,  # ISO timestamp or None for now
    "source": "auto",  # auto prefers mm_events full orderbook; falls back to mm_market_facts
}

MIN_UPDATES_BEFORE_SCORE = 1
ROLLING_WINDOW = 100

OUTPUT_DIR = REPO_ROOT / "research" / "model_lab" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_CONFIG

In [ ]:
until = (
    datetime.fromisoformat(DATA_CONFIG["until"].replace("Z", "+00:00"))
    if DATA_CONFIG["until"]
    else datetime.now(timezone.utc)
)
since = parse_since(DATA_CONFIG["since"], until=until)

steps = load_interval_steps(
    exchange=DATA_CONFIG["exchange"],
    symbol=DATA_CONFIG["symbol"],
    interval_ms=DATA_CONFIG["interval_ms"],
    since=since,
    until=until,
    source=DATA_CONFIG["source"],
)

if len(steps) < 2:
    raise RuntimeError(f"Need at least 2 steps, got {len(steps)}")

volumes = [step.total_volume for step in steps]
mid_values = [step.mid for step in steps]
spreads = [step.spread for step in steps]
zero_fraction = sum(1 for value in volumes if value <= 0.0) / len(volumes)

summary = {
    "n_steps": len(steps),
    "first_ts": steps[0].ts,
    "last_ts": steps[-1].ts,
    "zero_volume_fraction": zero_fraction,
    "avg_total_volume": sum(volumes) / len(volumes),
    "min_mid": min(mid_values),
    "max_mid": max(mid_values),
    "avg_spread": sum(spreads) / len(spreads),
}
summary

In [ ]:
def spec_from(kind: str, **config) -> str:
    if not config:
        return kind
    compact = json.dumps(config, separators=(",", ":"))
    return f"{kind}{compact}"


def evaluate_specs(specs: list[str], *, label_prefix: str = ""):
    results = {}
    for spec in specs:
        label = f"{label_prefix}{spec}"
        model = build_model(spec)
        result = run_walk_forward(
            model,
            steps,
            min_updates_before_score=MIN_UPDATES_BEFORE_SCORE,
        )
        results[label] = result
        summary = result.summary
        print(
            f"{label:90s} mean_nll={summary.mean_nll if summary else None} "
            f"median_nll={summary.median_nll if summary else None} scored={summary.n_scored if summary else 0}"
        )
    return results


def show_rows(rows, *, max_rows: int = 25):
    if pd is not None:
        return pd.DataFrame(rows).head(max_rows)
    return rows[:max_rows]

In [ ]:
baseline_specs = [
    "baseline:zero_flow",
    "baseline:mid_random_walk",
    spec_from("flow:hurdle_lognormal", decay=0.95, init_p0=0.3, init_mu=0.0, init_sigma=1.0),
    spec_from("price:conditional_mechanical_mixture", decay=0.99, reference_vwap_vol=1000.0, shape_alpha=0.5, student_df=3.0),
    spec_from("price:residual", residual_decay=0.95, residual_sigma=50.0, reference_vwap_vol=1000.0),
]

baseline_results = evaluate_specs(baseline_specs)
show_rows(leaderboard(baseline_results))

## Flow Sweeps

These runs compare existing production-compatible flow families and hyperparameters. Flow NLL is evaluated only against other flow models.

In [ ]:
flow_specs = []
flow_kinds = ["hurdle_lognormal", "hurdle_gamma", "hurdle_weibull", "gamma", "weibull"]
decays = [0.80, 0.90, 0.95, 0.975, 0.99, 0.995]
for kind in flow_kinds:
    for decay in decays:
        flow_specs.append(spec_from(f"flow:{kind}", decay=decay, init_p0=0.3, init_mu=0.0, init_sigma=1.0))

for init_p0 in [0.1, 0.3, 0.5, 0.6, 0.7]:
    flow_specs.append(spec_from("flow:hurdle_lognormal", decay=0.95, init_p0=init_p0, init_mu=0.0, init_sigma=1.0))

flow_results = evaluate_specs(flow_specs)
flow_leaderboard = [row for row in leaderboard(flow_results) if row["family"] == "flow"]
show_rows(flow_leaderboard, max_rows=20)

In [ ]:
flow_strata = stratified_nll(flow_results, steps)
show_rows(flow_strata, max_rows=40)

## Price Sweeps

Price NLL is compared within price targets. The production conditional mechanical mixture and residual models are full transition models; mid-delta baselines are narrower target baselines and should be interpreted separately.

In [ ]:
price_specs = []
for decay in [0.95, 0.975, 0.99, 0.995]:
    for reference_vwap_vol in [300.0, 500.0, 1000.0, 2000.0]:
        for shape_alpha in [0.3, 0.5, 0.7, 1.0]:
            price_specs.append(
                spec_from(
                    "price:conditional_mechanical_mixture",
                    decay=decay,
                    reference_vwap_vol=reference_vwap_vol,
                    shape_alpha=shape_alpha,
                    student_df=3.0,
                    initial_sigma=50.0,
                    initial_log_sigma=0.25,
                )
            )

for residual_decay in [0.80, 0.90, 0.95, 0.975, 0.99, 0.995]:
    for residual_sigma in [25.0, 50.0, 100.0, 200.0]:
        price_specs.append(
            spec_from(
                "price:residual",
                residual_decay=residual_decay,
                residual_sigma=residual_sigma,
                reference_vwap_vol=1000.0,
            )
        )

for decay in [0.90, 0.95, 0.99, 0.995]:
    for initial_sigma in [25.0, 50.0, 100.0, 200.0]:
        price_specs.append(spec_from("baseline:mid_random_walk", decay=decay, initial_sigma=initial_sigma))

price_results = evaluate_specs(price_specs)
price_leaderboard = [row for row in leaderboard(price_results) if row["family"] == "price"]
show_rows(price_leaderboard, max_rows=25)

In [ ]:
price_strata = stratified_nll(price_results, steps)
show_rows(price_strata, max_rows=50)

## Objective Comparison Diagnostics

These cells compare candidates on the same timestamps. Negative `mean_delta_nll` means the candidate beats the baseline on average.

In [ ]:
flow_baseline = spec_from("flow:hurdle_lognormal", decay=0.95, init_p0=0.3, init_mu=0.0, init_sigma=1.0)
price_baseline = spec_from("price:conditional_mechanical_mixture", decay=0.99, reference_vwap_vol=1000.0, shape_alpha=0.5, student_df=3.0)

flow_pairwise = paired_nll_deltas(flow_results, baseline_label=flow_baseline)
price_pairwise = paired_nll_deltas(price_results, baseline_label=price_baseline)

print("Flow paired deltas vs production hurdle lognormal")
display(show_rows(flow_pairwise, max_rows=20)) if pd is not None else print(flow_pairwise[:20])

print("Price paired deltas vs production conditional mixture")
display(show_rows(price_pairwise, max_rows=20)) if pd is not None else print(price_pairwise[:20])

In [ ]:
def plot_rolling(results, labels, *, title):
    if plt is None:
        print("matplotlib is not installed; skipping plot")
        return
    plt.figure(figsize=(12, 5))
    for label in labels:
        if label not in results:
            continue
        series = rolling_mean_nll(results[label], window=ROLLING_WINDOW)
        if not series:
            continue
        xs, ys = zip(*series)
        plt.plot(xs, ys, label=label[:80])
    plt.title(title)
    plt.xlabel("step index")
    plt.ylabel(f"rolling mean NLL, window={ROLLING_WINDOW}")
    plt.legend(loc="best", fontsize=8)
    plt.grid(True, alpha=0.2)
    plt.show()

best_flow_labels = [row["label"] for row in flow_leaderboard[:5]]
best_price_labels = [row["label"] for row in price_leaderboard[:5]]
plot_rolling(flow_results, best_flow_labels, title="Best flow models: rolling NLL")
plot_rolling(price_results, best_price_labels, title="Best price models: rolling NLL")

In [ ]:
def cumulative_delta_series(candidate_result, baseline_result):
    baseline = {
        score.step_index: float(score.nll)
        for score in baseline_result.scores
        if score.nll is not None and math.isfinite(float(score.nll))
    }
    total = 0.0
    series = []
    for score in candidate_result.scores:
        if score.nll is None or score.step_index not in baseline:
            continue
        value = float(score.nll)
        if not math.isfinite(value):
            continue
        total += value - baseline[score.step_index]
        series.append((score.step_index, total))
    return series


def plot_cumulative_deltas(results, labels, baseline_label, *, title):
    if plt is None:
        print("matplotlib is not installed; skipping plot")
        return
    plt.figure(figsize=(12, 5))
    baseline = results[baseline_label]
    for label in labels:
        if label == baseline_label or label not in results:
            continue
        series = cumulative_delta_series(results[label], baseline)
        if not series:
            continue
        xs, ys = zip(*series)
        plt.plot(xs, ys, label=label[:80])
    plt.axhline(0.0, color="black", linewidth=1)
    plt.title(title)
    plt.xlabel("step index")
    plt.ylabel("cumulative candidate NLL - baseline NLL")
    plt.legend(loc="best", fontsize=8)
    plt.grid(True, alpha=0.2)
    plt.show()

plot_cumulative_deltas(flow_results, best_flow_labels, flow_baseline, title="Flow cumulative NLL delta vs baseline")
plot_cumulative_deltas(price_results, best_price_labels, price_baseline, title="Price cumulative NLL delta vs baseline")

In [ ]:
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
summary_payload = {
    "run_id": run_id,
    "data_config": DATA_CONFIG,
    "n_steps": len(steps),
    "baseline_leaderboard": leaderboard(baseline_results),
    "flow_leaderboard": flow_leaderboard,
    "price_leaderboard": price_leaderboard,
    "flow_pairwise_vs_baseline": flow_pairwise,
    "price_pairwise_vs_baseline": price_pairwise,
    "flow_strata": flow_strata,
    "price_strata": price_strata,
}
summary_path = OUTPUT_DIR / f"{run_id}_online_model_search_summary.json"
summary_path.write_text(json.dumps(summary_payload, default=str, indent=2), encoding="utf-8")
summary_path

## Research-Only Candidate Models

These models live under `research/model_lab/models/experimental.py` and do not modify production runtime code.

Flow candidates:

- `flow:hurdle_log_student_t`: same online structure as hurdle lognormal, but heavier positive-volume tails.
- `flow:bivariate_hurdle_lognormal`: explicit joint zero/non-zero pattern probabilities plus lognormal positive sizes.

Price candidates:

- `price:student_t_mid_delta`: heavy-tailed stationary mid-delta baseline.
- `price:flow_conditioned_mid_delta`: online RLS mid-delta model using lagged total flow, signed flow, and spread.

In [ ]:
experimental_flow_specs = []
for decay in [0.90, 0.95, 0.975, 0.99]:
    for df in [3.0, 5.0, 8.0]:
        experimental_flow_specs.append(
            spec_from("flow:hurdle_log_student_t", decay=decay, init_p0=0.3, init_mu=0.0, init_sigma=1.0, init_df=df)
        )
    experimental_flow_specs.append(
        spec_from("flow:bivariate_hurdle_lognormal", decay=decay, init_mu=0.0, init_sigma=1.0)
    )

experimental_price_specs = []
for decay in [0.90, 0.95, 0.99, 0.995]:
    for df in [3.0, 5.0, 8.0]:
        experimental_price_specs.append(spec_from("price:student_t_mid_delta", decay=decay, initial_sigma=50.0, df=df))
    experimental_price_specs.append(
        spec_from("price:flow_conditioned_mid_delta", decay=decay, variance_decay=decay, initial_sigma=50.0)
    )

experimental_results = evaluate_specs(experimental_flow_specs + experimental_price_specs)
experimental_leaderboard = leaderboard(experimental_results)
show_rows(experimental_leaderboard, max_rows=40)

In [ ]:
all_flow_results = {**flow_results, **{k: v for k, v in experimental_results.items() if v.summary and v.summary.family == "flow"}}
all_price_results = {**price_results, **{k: v for k, v in experimental_results.items() if v.summary and v.summary.family == "price"}}

all_flow_leaderboard = [row for row in leaderboard(all_flow_results) if row["family"] == "flow"]
all_price_leaderboard = [row for row in leaderboard(all_price_results) if row["family"] == "price"]

print("All flow candidates")
display(show_rows(all_flow_leaderboard, max_rows=20)) if pd is not None else print(all_flow_leaderboard[:20])

print("All price candidates")
display(show_rows(all_price_leaderboard, max_rows=20)) if pd is not None else print(all_price_leaderboard[:20])

In [ ]:
experimental_flow_strata = stratified_nll(all_flow_results, steps)
experimental_price_strata = stratified_nll(all_price_results, steps)

print("Flow strata including experimental candidates")
display(show_rows(experimental_flow_strata, max_rows=60)) if pd is not None else print(experimental_flow_strata[:60])

print("Price strata including experimental candidates")
display(show_rows(experimental_price_strata, max_rows=60)) if pd is not None else print(experimental_price_strata[:60])

## Stationarity Checks Before Neural Models

The current production models are adaptive because levels and activity are non-stationary. Before trying a GRU or Transformer, inspect transformed series by block:

- `delta_mid`
- log return
- spread-normalized mid delta
- volume normalized by recent EWMA activity

If block means and variances still shift materially, prefer adaptive probabilistic models over fixed-weight neural models unless more history is available.

In [ ]:
stationarity_rows = stationarity_report(steps, blocks=8)
show_rows(stationarity_rows, max_rows=20)

In [ ]:
if plt is not None and stationarity_rows:
    keys = ["delta_mid_std", "log_return_std", "spread_norm_delta_std", "normalized_volume_std"]
    fig, axes = plt.subplots(len(keys), 1, figsize=(10, 10), sharex=True)
    for ax, key in zip(axes, keys):
        ax.plot([row["block"] for row in stationarity_rows], [row[key] for row in stationarity_rows], marker="o")
        ax.set_ylabel(key)
        ax.grid(True, alpha=0.2)
    axes[-1].set_xlabel("time block")
    fig.suptitle("Stationarity diagnostics by time block")
    plt.tight_layout()
    plt.show()
else:
    print("matplotlib unavailable or no stationarity rows")

## Interpretation Checklist

Use the final tables to decide next steps:

- Prefer a model only if it improves paired NLL on the same timestamps, not just an unpaired average.
- Check zero-volume and high-volume regimes separately; a quiet-regime winner may not be useful for burst risk.
- Inspect rolling NLL: a model that loses early but wins after warm-up may need better priors, not rejection.
- For price models, keep target scope explicit: full VWAP/book transition likelihood and mid-only likelihood are not the same objective.
- Only try neural sequence models if stationary transforms are stable enough or substantially more history is loaded.

## Objective Comparison Layer

Use paired deltas on identical step indices. Negative `mean_delta_nll` means the candidate beat the baseline on average.